In [78]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from PIL import Image
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import resnet50,ResNet50_Weights

In [79]:
data_dir = Path("images")
dataset = []

for class_folder in data_dir.iterdir():
    if class_folder.name != 'images' and class_folder.is_dir():
        if class_folder.is_dir():
            label = class_folder.name

            for image_file in class_folder.iterdir():
                if image_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    dataset.append({
                        'image_path': str(image_file),
                        'file_name' : image_file.name,
                        'label': label
                    })

dataset = pd.DataFrame(dataset)
dataset.head()

,image_path,file_name,label
0,images\armature\armature-coil001.jpg,armature-coil001.jpg,armature
1,images\armature\armature-coil002.jpg,armature-coil002.jpg,armature
2,images\armature\armature-coil003.jpg,armature-coil003.jpg,armature
3,images\armature\armature-coil004.jpg,armature-coil004.jpg,armature
4,images\armature\armature-coil005.jpg,armature-coil005.jpg,armature


In [80]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
dataset['label'] = le.fit_transform(dataset['label'])

In [81]:
print(len(set(dataset['label'])))

36


In [82]:
from sklearn.model_selection import train_test_split
datatrain, datatest = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset['label'])
dataval, datatest = train_test_split(datatest, test_size=0.5, random_state=42, stratify=datatest['label'])

In [83]:
img_size = 224

In [84]:
transform = transforms.Compose([
  transforms.Resize((img_size, img_size)),
  transforms.ToTensor(),
  #transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [85]:
class TrainDataset(Dataset):
  def __init__(self, df, transform=None, path_col='image_path', label_col='label'):
    self.df = df
    self.transform = transform
    self.path_col = df[path_col].values
    self.labels = df[label_col].values
  
  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    path = str(self.path_col[index])
    img = Image.open(path).convert('RGB')
    if self.transform:
      img = self.transform(img)
    return img, torch.tensor(self.labels[index], dtype=torch.long)

class TestDataset(Dataset):
  def __init__(self, df, transform=None, path_col='image_path'):
    self.df = df
    self.transform = transform
    self.path_col = df[path_col].values
  
  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    path = str(self.path_col[index])
    img = Image.open(path).convert('RGB')
    if self.transform:
      img = self.transform(img)
    return img

In [86]:
train_dataset = TrainDataset(
    df=datatrain,
    transform=transform,
    label_col='label',
    path_col='image_path'
)

validation_dataset = TrainDataset(
    df=dataval,
    transform=transform,
    label_col='label',
    path_col='image_path'
)

test_dataset = TestDataset(
    df=datatest,
    transform=transform,
    path_col='image_path'
)

In [87]:
BATCH_SIZE=32
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0
)
validation_loader=DataLoader(
    validation_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0
)

In [88]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [89]:
class CNN_preantrenat(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes) 
        )

    def forward(self, x):
        return self.model(x)

model=CNN_preantrenat(num_classes=36).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.0003)

In [90]:
epochs=15
train_losses =[]
val_losses=[]
val_accuracies=[]

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    model.eval()
    running_val_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
  
            _, predicted = torch.max(outputs.data, 1) 

            
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
      
    avg_val_loss = running_val_loss / len(validation_loader)
    val_losses.append(avg_val_loss)
    
    avg_val_accuracy = correct_predictions / total_samples
    val_accuracies.append(avg_val_accuracy)
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} Validation Loss:{avg_val_loss:.4f} Validation Accuracy: {avg_val_accuracy:.4f}")

Epoch [1/15] Train Loss: 2.8926 Validation Loss:2.4376 Validation Accuracy: 0.2830
Epoch [2/15] Train Loss: 2.1593 Validation Loss:2.0840 Validation Accuracy: 0.3776
Epoch [3/15] Train Loss: 1.7572 Validation Loss:2.0774 Validation Accuracy: 0.4022
Epoch [4/15] Train Loss: 1.4283 Validation Loss:2.0651 Validation Accuracy: 0.4040
Epoch [5/15] Train Loss: 1.1699 Validation Loss:2.1890 Validation Accuracy: 0.4104
Epoch [6/15] Train Loss: 0.9355 Validation Loss:2.2343 Validation Accuracy: 0.4258
Epoch [7/15] Train Loss: 0.7165 Validation Loss:2.4022 Validation Accuracy: 0.4049
Epoch [8/15] Train Loss: 0.5790 Validation Loss:2.5987 Validation Accuracy: 0.4222
Epoch [9/15] Train Loss: 0.4487 Validation Loss:2.5348 Validation Accuracy: 0.4268
Epoch [10/15] Train Loss: 0.4019 Validation Loss:2.4119 Validation Accuracy: 0.4386
Epoch [11/15] Train Loss: 0.3329 Validation Loss:2.7149 Validation Accuracy: 0.4295
Epoch [12/15] Train Loss: 0.3095 Validation Loss:2.7956 Validation Accuracy: 0.4140
E